In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [7]:
# 1. 데이터 불러오기
df = pd.read_csv('C:/Users/user/Desktop/프로젝트데이터/만족도데이터전처리/노인만족도조사2022_원핫인코딩.csv', encoding='utf-8')

# 2. 타깃/피처 분리
y = df["A1_1(전체삶 만족도, 타깃 예정)"]
X = df.drop(columns=["A1_1(전체삶 만족도, 타깃 예정)"])

# 3. 변수 선택 (Lasso)
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X, y)
X = X.loc[:, lasso.coef_ != 0]

# 4. 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 5. 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb3 in position 7: invalid start byte

In [ ]:
# 6. 모델 정의 (튜닝 생략하고 고정값 사용)
model = RandomForestRegressor(
    n_estimators=800,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42
)

model.fit(X_train, y_train)

# 7. 예측 및 성능 평가
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("📈 R² (Train):", round(r2_train, 4))
print("📈 R² (Test):", round(r2_test, 4))
print("📉 RMSE (Train):", round(rmse_train, 4))
print("📉 RMSE (Test):", round(rmse_test, 4))


In [ ]:
# 8. 교차검증
cv_r2 = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')
cv_rmse = -cross_val_score(model, X_scaled, y, cv=5, scoring='neg_root_mean_squared_error')

print("🔁 교차검증 R² 목록:", np.round(cv_r2, 4).tolist())
print("📊 교차검증 평균 R²:", round(cv_r2.mean(), 4))
print("🔁 교차검증 RMSE 목록:", np.round(cv_rmse, 4).tolist())
print("📉 교차검증 평균 RMSE:", round(cv_rmse.mean(), 4))